# Diabetes Prediction & Progression Analysis
## A Comprehensive Machine Learning Project

**Objective**: Build predictive models to:
1. Predict diabetes disease progression (Regression)
2. Classify patients as diabetic or non-diabetic (Classification)
3. Provide actionable insights through data analysis

## Section 1: Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10

print("✓ All libraries imported successfully!")

FileNotFoundError: Could not find module 'c:\Users\harsh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\.libs\vcomp140.dll' (or one of its dependencies). Try using the full path with constructor syntax.

## Section 2: Load & Explore Data

In [ ]:
# Load the diabetes dataset
data = load_diabetes()

# Create DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target_progression'] = data.target

print(f"Dataset Shape: {df.shape}")
print(f"\nFeatures: {list(df.columns[:-1])}")
print(f"Target Variable: target_progression (diabetes progression 1-year after baseline)")
print(f"\nFirst 5 rows:")
display(df.head())

In [ ]:
# Dataset Information
print("Dataset Information:")
print(f"Total Samples: {len(df)}")
print(f"Total Features: {df.shape[1] - 1}")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"\nData Types:\n{df.dtypes}")

print(f"\nStatistical Summary:")
display(df.describe())

## Section 3: Create Binary Classification Target
We'll create a binary classification target to identify patients as diabetic (high progression) or non-diabetic (low progression)

In [ ]:
# Create binary classification target: 1 = High progression (Diabetic), 0 = Low progression (Non-diabetic)
median_progression = df['target_progression'].median()
df['is_diabetic'] = (df['target_progression'] > median_progression).astype(int)

print(f"Median Progression Value: {median_progression:.2f}")
print(f"\nClass Distribution:")
print(f"Non-diabetic (0): {(df['is_diabetic'] == 0).sum()} samples ({(df['is_diabetic'] == 0).sum()/len(df)*100:.1f}%)")
print(f"Diabetic (1): {(df['is_diabetic'] == 1).sum()} samples ({(df['is_diabetic'] == 1).sum()/len(df)*100:.1f}%)")

## Section 4: Exploratory Data Analysis (EDA)

In [ ]:
# 4.1: Feature Distributions
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Distribution of All Features', fontsize=16, fontweight='bold')
axes = axes.ravel()

for idx, col in enumerate(df.columns[:-2]):
    sns.histplot(df[col], kde=True, ax=axes[idx], color='steelblue')
    axes[idx].set_title(f'{col}')
    axes[idx].set_xlabel('')

# Target progression distribution
sns.histplot(df['target_progression'], kde=True, ax=axes[10], color='coral')
axes[10].set_title('Target Progression')

axes[11].remove()
plt.tight_layout()
plt.show()

print("✓ Feature distributions plotted")

In [ ]:
# 4.2: Correlation Matrix
plt.figure(figsize=(12, 8))
correlation_matrix = df.drop('is_diabetic', axis=1).corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - All Features & Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 5 Features Correlated with Target Progression:")
target_corr = correlation_matrix['target_progression'].sort_values(ascending=False)
display(target_corr.head(6))

In [ ]:
# 4.3: Feature vs Target Relationships
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Feature vs Target Progression Relationship', fontsize=16, fontweight='bold')
axes = axes.ravel()

for idx, col in enumerate(df.columns[:-2]):
    axes[idx].scatter(df[col], df['target_progression'], alpha=0.5, color='steelblue')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Target Progression')
    axes[idx].set_title(f'{col} vs Target')

axes[11].remove()
plt.tight_layout()
plt.show()

print("✓ Feature relationships plotted")

## Section 5: Data Preparation

In [ ]:
# Prepare features and targets
X = df.drop(['target_progression', 'is_diabetic'], axis=1)
y_regression = df['target_progression']  # For regression: continuous progression values
y_classification = df['is_diabetic']      # For classification: binary diabetic status

# Split data: 80% train, 20% test
X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_regression, test_size=0.2, random_state=42
)
_, _, y_clf_train, y_clf_test = train_test_split(
    X, y_classification, test_size=0.2, random_state=42
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training Set: {X_train_scaled.shape}")
print(f"Test Set: {X_test_scaled.shape}")
print(f"✓ Data prepared and scaled")

## Section 6: Regression Models (Diabetes Progression Prediction)

In [ ]:
# Train regression models
print("Training Regression Models...\n")

# 1. Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_reg_train)
y_pred_lr = lr_model.predict(X_test_scaled)
print("✓ Linear Regression trained")

# 2. Ridge Regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_reg_train)
y_pred_ridge = ridge_model.predict(X_test_scaled)
print("✓ Ridge Regression trained")

# 3. Lasso Regression
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train_scaled, y_reg_train)
y_pred_lasso = lasso_model.predict(X_test_scaled)
print("✓ Lasso Regression trained")

# 4. Random Forest
rf_reg_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_reg_model.fit(X_train_scaled, y_reg_train)
y_pred_rf_reg = rf_reg_model.predict(X_test_scaled)
print("✓ Random Forest Regressor trained")

# 5. Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train_scaled, y_reg_train)
y_pred_gb = gb_model.predict(X_test_scaled)
print("✓ Gradient Boosting Regressor trained")

In [ ]:
# Evaluate regression models
regression_results = {}
models_reg = {
    'Linear Regression': (lr_model, y_pred_lr),
    'Ridge Regression': (ridge_model, y_pred_ridge),
    'Lasso Regression': (lasso_model, y_pred_lasso),
    'Random Forest': (rf_reg_model, y_pred_rf_reg),
    'Gradient Boosting': (gb_model, y_pred_gb)
}

print("="*70)
print(" REGRESSION MODELS PERFORMANCE: Predicting Diabetes Progression")
print("="*70)

for model_name, (model, y_pred) in models_reg.items():
    mse = mean_squared_error(y_reg_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_reg_test, y_pred)
    r2 = r2_score(y_reg_test, y_pred)
    
    regression_results[model_name] = {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}
    
    print(f"\n{model_name}:")
    print(f"  MSE:  {mse:>10.2f}")
    print(f"  RMSE: {rmse:>10.2f}")
    print(f"  MAE:  {mae:>10.2f}")
    print(f"  R²:   {r2:>10.4f}")

print("\n" + "="*70)

In [ ]:
# Compare regression models
results_df = pd.DataFrame(regression_results).T
print("\nRegression Models Comparison (sorted by R²):")
display(results_df.sort_values('R2', ascending=False))

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² Scores
axes[0].barh(results_df.index, results_df['R2'], color='steelblue')
axes[0].set_xlabel('R² Score')
axes[0].set_title('Model Performance: R² Scores')
axes[0].invert_yaxis()

# RMSE
axes[1].barh(results_df.index, results_df['RMSE'], color='coral')
axes[1].set_xlabel('RMSE')
axes[1].set_title('Model Performance: RMSE (lower is better)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## Section 7: Classification Models (Diabetic Status Prediction)

In [ ]:
# Train classification models
print("Training Classification Models...\n")

# 1. Logistic Regression
log_reg_model = LogisticRegression(max_iter=1000, random_state=42)
log_reg_model.fit(X_train_scaled, y_clf_train)
y_pred_log_reg = log_reg_model.predict(X_test_scaled)
print("✓ Logistic Regression trained")

# 2. Random Forest Classifier
rf_clf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf_model.fit(X_train_scaled, y_clf_train)
y_pred_rf_clf = rf_clf_model.predict(X_test_scaled)
print("✓ Random Forest Classifier trained")

In [ ]:
# Evaluate classification models
classification_results = {}
models_clf = {
    'Logistic Regression': y_pred_log_reg,
    'Random Forest Classifier': y_pred_rf_clf
}

print("="*70)
print(" CLASSIFICATION MODELS PERFORMANCE: Predicting Diabetic Status")
print("="*70)

for model_name, y_pred in models_clf.items():
    accuracy = accuracy_score(y_clf_test, y_pred)
    precision = precision_score(y_clf_test, y_pred)
    recall = recall_score(y_clf_test, y_pred)
    f1 = f1_score(y_clf_test, y_pred)
    
    classification_results[model_name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    }
    
    print(f"\n{model_name}:")
    print(f"  Accuracy:  {accuracy:>10.4f}")
    print(f"  Precision: {precision:>10.4f}")
    print(f"  Recall:    {recall:>10.4f}")
    print(f"  F1-Score:  {f1:>10.4f}")

print("\n" + "="*70)

In [ ]:
# Detailed classification reports
print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORTS")
print("="*70)

for model_name, y_pred in models_clf.items():
    print(f"\n{model_name}:")
    print(classification_report(y_clf_test, y_pred, target_names=['Non-Diabetic', 'Diabetic']))

In [ ]:
# Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Confusion Matrices - Classification Models', fontsize=14, fontweight='bold')

for idx, (model_name, y_pred) in enumerate(models_clf.items()):
    cm = confusion_matrix(y_clf_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Non-Diabetic', 'Diabetic'],
                yticklabels=['Non-Diabetic', 'Diabetic'])
    axes[idx].set_title(model_name)
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

## Section 8: Actual vs Predicted Visualizations

In [ ]:
# Actual vs Predicted for Regression
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Regression Models: Actual vs Predicted Diabetes Progression', fontsize=14, fontweight='bold')

predictions_reg = [
    ('Linear Regression', y_pred_lr),
    ('Ridge Regression', y_pred_ridge),
    ('Lasso Regression', y_pred_lasso),
    ('Random Forest', y_pred_rf_reg),
    ('Gradient Boosting', y_pred_gb)
]

for idx, (name, y_pred) in enumerate(predictions_reg):
    ax = axes[idx // 3, idx % 3]
    ax.scatter(y_reg_test, y_pred, alpha=0.5, color='steelblue')
    ax.plot([y_reg_test.min(), y_reg_test.max()], 
            [y_reg_test.min(), y_reg_test.max()], 
            'r--', lw=2, label='Perfect Prediction')
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{name} (R² = {r2_score(y_reg_test, y_pred):.3f})')
    ax.legend()

# Remove extra subplot
axes[1, 2].remove()
plt.tight_layout()
plt.show()

## Section 9: Feature Importance Analysis

In [ ]:
# Feature importance from best models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Importance Analysis', fontsize=14, fontweight='bold')

# Random Forest Regressor
feature_importance_rf = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_reg_model.feature_importances_
}).sort_values('Importance', ascending=False)

axes[0].barh(feature_importance_rf['Feature'], feature_importance_rf['Importance'], color='steelblue')
axes[0].set_xlabel('Importance')
axes[0].set_title('Random Forest Regressor - Feature Importance')
axes[0].invert_yaxis()

# Random Forest Classifier
feature_importance_clf = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_clf_model.feature_importances_
}).sort_values('Importance', ascending=False)

axes[1].barh(feature_importance_clf['Feature'], feature_importance_clf['Importance'], color='coral')
axes[1].set_xlabel('Importance')
axes[1].set_title('Random Forest Classifier - Feature Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print("\nTop 5 Important Features (Regression):")
display(feature_importance_rf.head())
print("\nTop 5 Important Features (Classification):")
display(feature_importance_clf.head())

## Section 10: Make Predictions on New Patients

In [ ]:
# Example predictions on test set
print("="*80)
print(" EXAMPLE PREDICTIONS: Best Models")
print("="*80)

# Using best regression model (Gradient Boosting) and best classifier (Random Forest)
predictions_summary = pd.DataFrame({
    'Actual_Progression': y_reg_test.values,
    'Predicted_Progression_GB': y_pred_gb,
    'Actual_Diabetic': y_clf_test.values,
    'Predicted_Diabetic_RF': y_pred_rf_clf
})

# Add interpretation
predictions_summary['Diabetic_Status'] = predictions_summary['Predicted_Diabetic_RF'].map(
    {0: 'Non-Diabetic', 1: 'Diabetic'}
)
predictions_summary['Risk_Level'] = pd.cut(
    predictions_summary['Predicted_Progression_GB'],
    bins=3,
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

print(f"\nShowing first 20 predictions:")
display(predictions_summary.head(20))

print(f"\nPrediction Summary Statistics:")
print(f"Average Predicted Progression: {predictions_summary['Predicted_Progression_GB'].mean():.2f}")
print(f"Predicted Diabetic (1): {(predictions_summary['Predicted_Diabetic_RF'] == 1).sum()} patients")
print(f"Predicted Non-Diabetic (0): {(predictions_summary['Predicted_Diabetic_RF'] == 0).sum()} patients")

## Section 11: Export Data

In [ ]:
import os

# Create export directory
export_dir = r'd:\Machine Learning\Projects\Phase_1\Exports'
os.makedirs(export_dir, exist_ok=True)

# Export 1: Full Dataset
df.to_csv(f'{export_dir}\\diabetes_dataset_full.csv', index=False)
print(f"✓ Full dataset exported: {export_dir}\\diabetes_dataset_full.csv")

# Export 2: Training Set
train_data = pd.DataFrame(X_train_scaled, columns=X.columns)
train_data['target_progression'] = y_reg_train.values
train_data['is_diabetic'] = y_clf_train.values
train_data.to_csv(f'{export_dir}\\diabetes_training_set.csv', index=False)
print(f"✓ Training set exported: {export_dir}\\diabetes_training_set.csv")

# Export 3: Test Set with Predictions
test_data = pd.DataFrame(X_test_scaled, columns=X.columns)
test_data['actual_progression'] = y_reg_test.values
test_data['predicted_progression'] = y_pred_gb
test_data['actual_diabetic'] = y_clf_test.values
test_data['predicted_diabetic'] = y_pred_rf_clf
test_data.to_csv(f'{export_dir}\\diabetes_test_set_with_predictions.csv', index=False)
print(f"✓ Test set with predictions exported: {export_dir}\\diabetes_test_set_with_predictions.csv")

# Export 4: Predictions Summary
predictions_summary.to_csv(f'{export_dir}\\predictions_summary.csv', index=False)
print(f"✓ Predictions summary exported: {export_dir}\\predictions_summary.csv")

# Export 5: Model Performance Report
performance_report = pd.DataFrame(regression_results).T
performance_report.to_csv(f'{export_dir}\\regression_model_performance.csv')
print(f"✓ Model performance exported: {export_dir}\\regression_model_performance.csv")

print(f"\n✓ All data files exported successfully!")

## Section 12: Summary & Conclusions

In [ ]:
print("\n" + "="*80)
print(" PROJECT SUMMARY & KEY FINDINGS")
print("="*80)

print("\n📊 DATA OVERVIEW:")
print(f"  • Total Samples: {len(df)}")
print(f"  • Features: {X.shape[1]}")
print(f"  • Missing Values: 0")
print(f"  • Target Range (Progression): {df['target_progression'].min():.1f} - {df['target_progression'].max():.1f}")

print("\n🎯 REGRESSION RESULTS (Diabetes Progression Prediction):")
best_reg_model = results_df.sort_values('R2', ascending=False).index[0]
best_reg_r2 = results_df.sort_values('R2', ascending=False)['R2'].iloc[0]
best_reg_rmse = results_df.sort_values('R2', ascending=False)['RMSE'].iloc[0]
print(f"  • Best Model: {best_reg_model}")
print(f"  • R² Score: {best_reg_r2:.4f}")
print(f"  • RMSE: {best_reg_rmse:.2f}")

print("\n🏥 CLASSIFICATION RESULTS (Diabetic Status Prediction):")
clf_results_df = pd.DataFrame(classification_results).T
best_clf_model = clf_results_df.sort_values('Accuracy', ascending=False).index[0]
best_clf_acc = clf_results_df.sort_values('Accuracy', ascending=False)['Accuracy'].iloc[0]
print(f"  • Best Model: {best_clf_model}")
print(f"  • Accuracy: {best_clf_acc:.4f}")

print("\n📈 TOP 3 PREDICTIVE FEATURES:")
for idx, row in feature_importance_rf.head(3).iterrows():
    print(f"  {idx+1}. {row['Feature']}: {row['Importance']:.4f}")

print("\n✅ MODELS TRAINED:")
print("  Regression: Linear, Ridge, Lasso, Random Forest, Gradient Boosting")
print("  Classification: Logistic Regression, Random Forest")

print("\n💾 DATA EXPORTED:")
print(f"  • Full Dataset")
print(f"  • Training Set")
print(f"  • Test Set with Predictions")
print(f"  • Predictions Summary")
print(f"  • Model Performance Report")

print("\n" + "="*80)